# 因子固定绝对评分

将 RankIC、ICIR、分层表现、近期 IC 和衰减保持度统一为 100 分制，并对创业板与中证 800 使用不同固定阈值。

In [1]:
from pathlib import Path
import sys
import pandas as pd

项目目录 = Path.cwd()
if not (项目目录 / 'outputs').exists():
    项目目录 = 项目目录.parent
sys.path.insert(0, str(项目目录))

from src.factor_scoring import combine_pools, prepare_scoring_inputs, score_pool

## 1. 整理评分输入

从研究流水线中提取 5 日全样本指标、年度 IC、分层单调性和头尾收益差。

In [2]:
汇总 = pd.read_csv(项目目录 / 'outputs/factor_summary.csv')
日度 = pd.read_csv(项目目录 / 'outputs/daily_factor_results.csv')
全样本指标, 年度IC = prepare_scoring_inputs(汇总, 日度, horizon=5)
全样本指标.round(4)

,factor_name,rank_ic_mean,rank_ic_ir,rank_ic_positive_ratio,layer_spearman_mean,top_bottom_spread_mean,spread_positive_ratio
0,momentum_20d,-0.0476,-0.4204,0.3344,-0.1371,-0.0050,0.3530
1,reversal_5d,0.0889,0.7959,0.7923,0.2456,0.0086,0.7447
2,volatility_20d,0.0050,0.0440,0.5166,0.0159,0.0007,0.5254
3,turnover_signal,0.0007,0.0059,0.5002,0.0010,0.0000,0.4980


## 2. 分股票池评分

总分 = 基础有效性 70 分 + 近期表现 20 分 + 衰减保持度 10 分；硬筛选检查方向、近期稳定性、Retention 和最低分数。

In [3]:
创业板 = score_pool(全样本指标, 年度IC, pool='chinext')
中证800 = score_pool(全样本指标, 年度IC, pool='csi800')

展示列 = ['factor_name', 'pool_score_100', 'pool_grade', 'pool_pass', 'fail_reason']
创业板[展示列].round(2)

,factor_name,pool_score_100,pool_grade,pool_pass,fail_reason
0,reversal_5d,99.60,A,True,通过
1,momentum_20d,77.96,B,True,通过
2,volatility_20d,0.00,D,False,方向不明确
3,turnover_signal,0.00,D,False,方向不明确


## 3. 双股票池综合筛选

只有跨池方向一致、两池近期稳定、Retention 达标且两池评分均不低于 60 分的因子才通过。

In [4]:
综合 = combine_pools(创业板, 中证800)
综合[['factor_name', 'final_score', 'direction_consistent', 'final_pass', 'grade', 'final_rank']].round(2)

,factor_name,final_score,direction_consistent,final_pass,grade,final_rank
0,reversal_5d,99.60,True,True,A,1
1,momentum_20d,88.98,True,True,A,2
2,volatility_20d,0.00,False,False,D,3
3,turnover_signal,0.00,False,False,D,3


## 说明

- 固定阈值不会随新增因子变化，便于跨批次比较。
- 示例结果来自合成数据，只用于展示评分程序。
- 实际研究中应结合样本外检验、交易成本和因子相关性进一步筛选。